# Leha GPU Voice Clone - Colab or Kaggle

This notebook generates cloned Leha voice samples from your reference audio using a GPU. Use only audio you own or have permission to clone.

Steps:
1. On your laptop run `python tools/prepare_voice_gpu_bundle.py`.
2. Upload `voice_gpu_bundle.zip` here.
3. In Colab: Runtime > Change runtime type > GPU.
4. In Kaggle: Notebook Settings > Accelerator > GPU.
5. Run all cells and download `leha_gpu_outputs.zip`.

In [ ]:
!nvidia-smi
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip -q install chatterbox-tts soundfile

Upload `voice_gpu_bundle.zip`. In Kaggle, add it as input data or upload it into the notebook working directory.

In [ ]:
from pathlib import Path
import json, zipfile, shutil, os

bundle = Path('voice_gpu_bundle.zip')
if not bundle.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            bundle = Path(next(iter(uploaded)))
    except Exception:
        pass

if not bundle.exists():
    kaggle_matches = list(Path('/kaggle/input').glob('**/voice_gpu_bundle.zip')) if Path('/kaggle/input').exists() else []
    if kaggle_matches:
        bundle = kaggle_matches[0]

assert bundle.exists(), 'Upload voice_gpu_bundle.zip first.'
work = Path('leha_voice_work')
if work.exists():
    shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(work)
manifest = json.loads((work / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2))
refs = sorted((work / 'refs').glob('*'))
print('refs:', refs)

In [ ]:
from chatterbox.tts import ChatterboxTTS
import torchaudio as ta
import torch, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChatterboxTTS.from_pretrained(device=device)
print('loaded on', device)

Pick the reference file. `leha_reference_mix.wav` is usually best because it combines both uploaded audios.

In [ ]:
reference = work / 'refs' / manifest.get('recommended_reference', refs[0].name)
if not reference.exists():
    reference = refs[0]
print('using reference:', reference)

In [ ]:
out_dir = Path('leha_gpu_outputs')
out_dir.mkdir(exist_ok=True)

prompts = [line.strip() for line in (work / 'prompts.txt').read_text().splitlines() if line.strip()]
settings = [
    {'name': 'balanced', 'exaggeration': 0.50, 'cfg_weight': 0.50, 'temperature': 0.80},
    {'name': 'clear', 'exaggeration': 0.35, 'cfg_weight': 0.65, 'temperature': 0.70},
    {'name': 'expressive', 'exaggeration': 0.70, 'cfg_weight': 0.40, 'temperature': 0.90},
]

for setting in settings:
    for i, text in enumerate(prompts, start=1):
        start = time.time()
        wav = model.generate(
            text,
            audio_prompt_path=str(reference),
            exaggeration=setting['exaggeration'],
            cfg_weight=setting['cfg_weight'],
            temperature=setting['temperature'],
        )
        path = out_dir / f"leha_{setting['name']}_{i:02d}.wav"
        ta.save(str(path), wav, model.sr)
        print(path, f'{time.time() - start:.2f}s')

In [ ]:
from IPython.display import Audio, display
for path in sorted(out_dir.glob('*.wav'))[:9]:
    print(path.name)
    display(Audio(str(path)))

When one setting sounds best, put its values into `jarvis_ai/config.py` for `CLONE_TTS_EXAGGERATION`, `CLONE_TTS_CFG_WEIGHT`, and `CLONE_TTS_TEMPERATURE`.

In [ ]:
import zipfile
zip_path = Path('leha_gpu_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(reference, 'reference_used.wav')
    zf.writestr('settings.json', json.dumps(settings, indent=2))
    for path in sorted(out_dir.glob('*.wav')):
        zf.write(path, path.name)
print('download:', zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('Kaggle: download from the right-side Output files panel.')